**Improvements to use multiple tools**
> - Ensure `add_user_message` and `add_assistant_message` can handle multiple message blocks
> - Allow `chat()` to receive a list of tools, and return fully generated message, not just text
> - Add a new `text_from_message()` function to extract all text from text blocks of the message
> - Add support for multiple tool calls in a conversation

In [49]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helper functions

> Note:
> - We are using `Message` package to make the message handling more robust/flexible.
> - Earlier, we had to parse through the response/message output to get the exact content, but using this package makes the solution more flexible, i.e. it can handle a string, a list of blocks, or message objects. Whether it is `add_assistant_msg(response, message)` or `add_assistant_msg(response, "Hi there")`, etc.

**Changes to make**
- `chat()`
    - Add `tools` parameter
    - Return the whole message, not just the text content
- `add_user_message()` & `add_assistant_message()`
    - Update `content` to handle this dynamic message type

In [ ]:
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        # Allow passing either a string or a Message object
        "content": message.content if isinstance(message, Message) else message,
        # TIP: You can find a better alternative with input validation in `./3_Streaming_with_Fine-grained_Tool_Call.ipynb`
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        # Easy extraction of content regardless of whether it's a Message object or a string
        "content": message.content if isinstance(message, Message) else message, 
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

### Extract text from message

- Parse & extract all blocks where the `type='text'`

In [51]:
def text_from_message(message):
    # Extract all text blocks in a message and concatenate them
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

## Tools: Functions & Schema

### Tool Function - Add duration to DateTime

In [52]:


from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

### Tool Function - Set Reminder

In [53]:
def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

### Tool Function - Get Current Datetime in User-Specified Format

In [ ]:
from anthropic.types import ToolParam

def get_current_datetime(dateformat="%Y-%m-%d %H:%M:%S"):
    # Input validation
    if not dateformat:
        raise ValueError("Date format string cannot be empty.")
    return datetime.now().strftime(dateformat)

### Tool Schema

In [ ]:
add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and/or time formatted as a string. Use this tool when the user asks what time or date it is, or needs a timestamp in a specific format. The format string uses Python's strftime directives (e.g., '%Y-%m-%d', '%H:%M:%S', '%A %B %d, %Y').",
  "input_schema": {
    "type": "object",
    "properties": {
      "dateformat": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls the structure of the returned datetime string. Defaults to '%Y-%m-%d %H:%M:%S' (e.g., '2025-03-29 14:30:00'). Must be a non-empty string. Common examples: '%Y-%m-%d' for date only, '%H:%M:%S' for time only, '%A %B %d, %Y' for a human-readable date.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

## Sample Message Call

In [ ]:
# messages = []

# add_user_message(messages, "What is the time in hour, minute, second format'?")

# response = client.messages.create(
#     model=model,
#     max_tokens=1000,
#     messages=messages,
#     tools=[get_current_datetime_schema],
# )

# add_assistant_message(messages, response) # Assistant is able to handle if I pass the whole response object or just the content

# messages

[{'role': 'user',
  'content': "What is the time in hour, minute, second format'?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01BLiNq4VLgdeVEMzSoyUkAW', caller=DirectCaller(type='direct'), input={'dateformat': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

## Multi-turn Conversation Tool Call Function

**Required steps**
- Run the conversation until LLM doesn't want to use any tool (possibly, the end of conversation where it gives a final response)
- In this function, allow LLM to call desired function of choice (could be many tool call in a single message)
- What to look for in `message.content`?
    - It will have `TextBlock` - what claude is trying to do (not important, if required print to user for info only using `text_from_message()`)
    - Fetch all the `ToolUseBlock` in the message content and get `id`, `input`, `name` for tool use
    - For each `ToolUseBlock` found,
        - Run specified tool with given inputs
        - Take output from tool, and put it into a `ToolResultBlock`
        - Return all `ToolResultBlock` as a list (if multiple) as a user message.

In [57]:
import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    if tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    if tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

def run_tools(message):
    tool_use_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_use in tool_use_requests:
        try:
            tool_output = run_tool(tool_use.name, tool_use.input) # Call the relevant tool function based on the tool name
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_use.id,  # Link the result back to the original tool use
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": f"Error occurred: {e}",
                "is_error": True
            }
        tool_result_blocks.append(tool_result_block)
    return tool_result_blocks

In [58]:
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema,
                ]
            ) # returns the whole message object

        add_assistant_message(messages, response) # this is designed to handle raw message object

        print("Assistant response:", text_from_message(response))

        if response.stop_reason != "tool_use": # Does LLM want to call a tool? If not, we can end the conversation
            break

        tool_results = run_tools(response) # Why? LLM might want to call more than one tool

        add_user_message(messages, tool_results) # Add the tool results back into the conversation for the LLM to see and respond to
    return messages

## Test Call

> Note:
> - `ToolUse` requests from LLM can be simultaneous, or sequential, depending on the requirement

**Message Flow**:
- User message with the request
- Assistant message containing both text and tool use blocks
- Tool result messages
- Follow-up assistant messages

#### 1 - Prompt that would request multiple `ToolUseBlock` in a single message

In [59]:
messages = []

add_user_message(messages, "What is the current time, and also tell me the current date in US format?")

run_conversation(messages)

Assistant response: I'll get the current time and date for you in US format.
Assistant response: The current time is **11:10:24 PM** and the current date in US format is **03/29/2026**.


[{'role': 'user',
  'content': 'What is the current time, and also tell me the current date in US format?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll get the current time and date for you in US format.", type='text'),
   ToolUseBlock(id='toolu_017Lw97y3Yu7eJNRVN4PPbQ2', caller=DirectCaller(type='direct'), input={'dateformat': '%I:%M:%S %p'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01TaoUYJDvfRvEjS7TKV2Hqr', caller=DirectCaller(type='direct'), input={'dateformat': '%m/%d/%Y'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_017Lw97y3Yu7eJNRVN4PPbQ2',
    'content': '"11:10:24 PM"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01TaoUYJDvfRvEjS7TKV2Hqr',
    'content': '"03/29/2026"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is **11:10:24 PM

#### 2 - Input a prompt which would make LLM ask for multiple tool calls (sequential)

In [60]:
messages = []

add_user_message(messages, "Set a reminder for me 7 days from now to 'Call Alice about the project update'")

run_conversation(messages)

Assistant response: I'll set a reminder for you 7 days from now with that message. Let me first get the current date and time, then calculate 7 days from now.
Assistant response: Now I'll calculate 7 days from now and set the reminder:
Assistant response: Now I'll set the reminder for that date and time:
----
Setting the following reminder for 2026-04-05T23:10:26:
Call Alice about the project update
----
Assistant response: Perfect! I've set a reminder for you on **Sunday, April 5, 2026 at 11:10 PM** to **"Call Alice about the project update"**. You'll receive a notification at that time.


[{'role': 'user',
  'content': "Set a reminder for me 7 days from now to 'Call Alice about the project update'"},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll set a reminder for you 7 days from now with that message. Let me first get the current date and time, then calculate 7 days from now.", type='text'),
   ToolUseBlock(id='toolu_01Q7ZyhwHXc6nt6Wodxw7cxE', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Q7ZyhwHXc6nt6Wodxw7cxE',
    'content': '"2026-03-29 23:10:26"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll calculate 7 days from now and set the reminder:", type='text'),
   ToolUseBlock(id='toolu_01BGYEjAA1RqU8RNYrcMhphr', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-03-29 23:10:26', 'duration': 7, 'unit': 'days', 'input_format': '%Y-%m-%d %H:%M